# behaviz — `hue` & `group` showcase
Exhaustive tour of per-category series, colors, legends, and dodging.

- **`hue`** -> one colored series per category + legend
- **`group`** -> one series per category, same color, no legend
- **both** -> per-(group x hue) primitive, colored by hue
- bars/errorbars **dodge** (`centered` default, or `stacked`)

Grouping needs `data=`; category args are column names.

In [ ]:
import numpy as np
import pandas as pd
import behaviz as bv
from bokeh.io import output_notebook, show
output_notebook()

def render(out):
    fig = out[0]
    if bv.get_renderer().name == 'bokeh':
        show(fig)



In [ ]:
rng = np.random.default_rng(7)
subjects = [f's{i}' for i in range(6)]
conds = ['control', 'drug A', 'drug B']
t = np.linspace(0, 1, 40)
rows = []
for ci, cond in enumerate(conds):
    for s in subjects:
        base = 0.6 * ci + 0.15 * rng.standard_normal()
        y = np.sin(t * 6) + base + 0.06 * rng.standard_normal(t.size)
        rows += [dict(t=ti, signal=yi, condition=cond, subject=s) for ti, yi in zip(t, y)]
df = pd.DataFrame(rows)
df_s0 = df[df.subject == 's0']

# mean +/- 95% CI band per condition (for fill_between)
band = df.groupby(['condition', 't'], as_index=False)['signal'].mean().rename(columns={'signal': 'mean'})
band['sem'] = df.groupby(['condition', 't'], as_index=False)['signal'].sem()['signal']
band['lo'] = band['mean'] - 1.96 * band['sem']
band['hi'] = band['mean'] + 1.96 * band['sem']

# aggregated bar data: conditions x sessions
bars = pd.DataFrame({
    'cond_idx':  [0, 1, 2, 0, 1, 2],
    'condition': ['control', 'drug A', 'drug B'] * 2,
    'session':   ['pre'] * 3 + ['post'] * 3,
    'score':     [3.1, 5.2, 2.4, 4.0, 1.8, 6.1],
    'sem':       [0.3, 0.4, 0.2, 0.35, 0.25, 0.5],
})
df

In [ ]:
bv.set_renderer('matplotlib')

## Overlay plots
`line`, `scatter`, `step`, `fill_between` layer colored series in place.

### `hue` — one colored line per condition (+ legend)

In [ ]:
render(bv.plot_line('t', 'signal', data=df, hue='condition'))

### `group` — one line per subject, same color, no legend (spaghetti)

In [ ]:
render(bv.plot_line('t', 'signal', data=df, group='subject'))

### `group` + `hue` — per-subject lines, colored by condition

In [ ]:
render(bv.plot_line('t', 'signal', data=df, group='subject', hue='condition'))

### `hue` scatter

In [ ]:
render(bv.plot_scatter('t', 'signal', data=df, hue='condition'))

### `hue` step (one subject)

In [ ]:
render(bv.plot_step('t', 'signal', data=df_s0, hue='condition'))

### `hue` fill_between — 95% CI band per condition

In [ ]:
render(bv.plot_fill_between('t', 'hi', 'lo', data=band, hue='condition', alpha=0.3))

## Dodge — bars & error bars sit side by side

### Grouped bars (`centered`, default)

In [ ]:
render(bv.plot_bar('cond_idx', 'score', data=bars, hue='session'))

### Stacked bars (`dodge='stacked'`)

In [ ]:
render(bv.plot_bar('cond_idx', 'score', data=bars, hue='session', dodge='stacked'))

### Grouped error bars

In [ ]:
render(bv.plot_errorbar('cond_idx', 'score', 'sem', data=bars, hue='session', capsize=4))

## Color & order control

### `hue_order` — fix the category order

In [ ]:
render(bv.plot_line('t', 'signal', data=df, hue='condition', hue_order=['drug B', 'drug A', 'control']))

### `palette` as a list

In [ ]:
render(bv.plot_line('t', 'signal', data=df, hue='condition', palette=['#1b9e77', '#d95f02', '#7570b3']))

### `palette` as a dict (per-category colors)

In [ ]:
render(bv.plot_line('t', 'signal', data=df, hue='condition',
                     palette={'control': '#888888', 'drug A': '#e41a1c', 'drug B': '#377eb8'}))

### `group_order`

In [ ]:
render(bv.plot_line('t', 'signal', data=df, group='condition'))

## Cross-backend — same call, every backend

### seaborn

In [ ]:
bv.set_renderer('seaborn')
render(bv.plot_line('t', 'signal', data=df, hue='condition'))

### bokeh (interactive)

In [ ]:
bv.set_renderer('bokeh')
render(bv.plot_scatter('t', 'signal', data=df, hue='condition'))

### bokeh stacked bars

In [ ]:
render(bv.plot_bar('cond_idx', 'score', data=bars, hue='session', dodge='stacked'))

In [ ]:
bv.set_renderer('matplotlib')  # reset